# SCRES-IA — Notebook final de David: ¿tu arquitectura le gana a RecurrentPPO?

**Objetivo:** poner *tu* arquitectura (DMLPA transformer, o la que quieras) dentro de varios
algoritmos de RL (**PPO, RecurrentPPO, SAC, QR-DQN, A2C, DQN**) y medir, en el **mismo entorno y
con la misma frontera** que usó RecurrentPPO en *Submission A*, si le ganas.

> **Cómo correrlo: simplemente `Runtime → Run all`.** Los defaults ya están puestos para una corrida
> real (`serious`, 200k timesteps = el mismo presupuesto que RecurrentPPO, celda `rho90_share90`, tu
> arquitectura `positional_v1_1` con PPO). No hace falta editar nada. En Colab, activa GPU
> (`Runtime → Change runtime type → GPU`) antes del Run all. Para un smoke de 2 minutos, cambia
> `RUN_PROFILE = "debug"` en la celda de config.

Este notebook es **apples-to-apples**: corre sobre el entorno de **Program Q** (`ProgramORetOnlyEnv`,
Discrete(4), 8 decisiones, recompensa = ReT canónica), evalúa contra la **frontera exacta de 65 536
calendarios de lazo abierto** y contra el **controlador clásico belief-MPC** — exactamente los dos
rivales del paper. (Tu notebook anterior corría en `dkana_thesis_faithful` con acción `Box(18)`, que
es *otro* entorno y **no** es comparable con RecurrentPPO. Este sí lo es.)

## ¿A qué tienes que ganarle? (los resultados que YA tenemos, sellados)

Dos cantidades importan (la "escalera"):
- **`H_learned` = tu modelo − mejor calendario de lazo abierto.** Mide si aprendes a reaccionar.
  RecurrentPPO ya le gana a los 65 536 con **+0.07 a +0.12**. Igualar esto es "table stakes".
- **`H_neural` = tu modelo − mejor clásico (belief-MPC).** *Esta es la que importa.* Es la **prima
  neural**. RecurrentPPO empató: **H_neural ≈ 0**. **Nadie ha superado al clásico todavía.**

| Celda | RecurrentPPO `H_learned` | RecurrentPPO **`H_neural`** | tu DMLPA previo `H_neural` |
|---|---:|---:|---:|
| rho75_share90 | +0.0795 | **−0.00159** (IC [−0.0063,+0.0031]) | −0.0069 |
| rho90_share75 | +0.0726 | **−0.00072** (IC [−0.0055,+0.0041]) | −0.0145 |
| rho90_share90 | +0.1172 | **−0.00041** (IC [−0.0027,+0.0019]) | +0.0034 |

> **La barra de victoria (para un paper fuerte):** no basta con `H_neural > 0` por ruido. Necesitas
> **`H_neural` con LCB95 ≥ +0.01** (el SESOI), en al menos una celda, **respetando los guardrails**
> (peor-producto no peor de −0.02, sin aumentar pedidos perdidos). Eso sería la primera prima neural
> real del programa. Tu DMLPA previo llegó a +0.0034 en una celda — *cerca de 0, pero no ganó*.

El notebook te lo dice explícitamente al final: **¿le ganaste a RecurrentPPO? ¿al clásico? ¿pasas la
barra?**


## Instrucciones (léelas una vez)

### Cómo abrir
- **Colab:** https://colab.research.google.com/github/Thom-320/scres-ia/blob/qr1-c1-natural-continuation/notebooks/scresia_david_multialgo_FINAL.ipynb
- **Kaggle:** https://www.kaggle.com/kernels/welcome?src=https://github.com/Thom-320/scres-ia/blob/qr1-c1-natural-continuation/notebooks/scresia_david_multialgo_FINAL.ipynb
  (si no importa solo: en Kaggle → *New Notebook → File → Import Notebook → Link* y pega esa URL de GitHub.)

### Requisitos por plataforma (importante)
- **Colab:** `Runtime → Change runtime type → GPU`. Luego `Runtime → Run all`.
- **Kaggle:** en el panel derecho activa **`Internet: On`** (obligatorio: sin internet no puede clonar
  el repo ni instalar paquetes) y **`Accelerator: GPU`**. Luego `Run All`.
- Si en Colab pip actualiza algún paquete base y aparece un aviso de reinicio, haz
  `Runtime → Restart runtime` y vuelve a `Run all` (no hace falta cambiar nada).

### Cómo correrlo
1. Activa GPU (arriba) y, en Kaggle, Internet On.
2. **`Run all`.** Nada más. Los defaults ya están listos.

### Tiempo estimado (default: PPO + tu transformer, `serious` = 200k timesteps, 64 tapes)
- **GPU (recomendado): ~25–35 min.**  |  **CPU: ~60–65 min.**
  (medido: PPO ~57 steps/s en CPU → 200k ≈ 59 min; baselines ≈ 3 min. En GPU el entrenamiento es
  varias veces más rápido; los baselines son ~3 min en cualquier caso.)
- **RecurrentPPO** ≈ 1.5× el tiempo de PPO. **SAC / DQN / QR-DQN** (off-policy con replay) son
  **más lentos**; para SAC en `serious`, considera GPU o baja `TOTAL_TIMESTEPS_SERIOUS`.

### ✅ Qué PUEDES cambiar (todo en la celda de **Configuración** y en la de **Arquitectura**)
- `ALGORITHM`: **`"PPO"`, `"RecurrentPPO"`, `"SAC"`, `"QRDQN"`, `"A2C"`, `"DQN"`** — todos ya
  probados con tu arquitectura. *Revisa que corra SAC y los algoritmos pertinentes de Stable-Baselines
  para este problema* (SAC/PPO/RecurrentPPO son los más relevantes) simplemente cambiando esta línea.
- `ARCH_VARIANT` y, sobre todo, **la celda “Tu arquitectura”**: edítala y revísala libremente. Hay un
  **chequeo de forma** que valida que tu `forward` devuelva `[batch, features_dim]` — si la editas y el
  chequeo pasa, tu arquitectura es válida para TODOS los algoritmos.
- `RUN_PROFILE` (`"serious"` real / `"debug"` smoke de ~2 min), `CELL_ID`, `FRAME_STACK`,
  `FEATURES_DIM`, `SEED`, `TOTAL_TIMESTEPS_*`, `EVAL_TAPES_*`, hiperparámetros por algoritmo.

### ⛔ Qué NO debes cambiar (rompe la comparación justa con RecurrentPPO)
- **El entorno** (`ProgramORetOnlyEnv`) — es EXACTAMENTE el que enfrentó RecurrentPPO. Cambiarlo hace
  que tus números no sean comparables.
- **Cómo se calculan la frontera de 65 536 y el clásico belief-MPC** (celda de baselines) — son los
  dos rivales oficiales.
- **Las constantes `FROZEN_RECURRENTPPO`** — son los resultados sellados que hay que batir.
- **La lógica del veredicto y la barra de victoria** (`WIN_BAR_H_NEURAL_LCB = 0.01`).
- **El namespace de seeds selladas** — usa solo el namespace de desarrollo (ya configurado).

> **⚠️ Sé MUY cuidadoso con lo que cambies.** Las celdas marcadas `# ⛔ NO CAMBIAR` mantienen la
> comparación honesta; si tocas una, tus resultados dejan de ser comparables con RecurrentPPO. Cambia
> solo lo marcado `# ✅ PUEDES CAMBIAR`.

### Resultado
Al final, el notebook imprime un **veredicto claro**: `[SÍ/NO]` le ganaste al lazo abierto, a
RecurrentPPO, al clásico, y si **pasaste la barra** de prima neural real. Si la pasas, lo dice
explícitamente.


## 1) Configuración  ·  ✅ **PUEDES CAMBIAR TODO ESTO**

In [ ]:

# ==================== CONFIG — listo para "Run All" (no hace falta editar) ====================
# Los defaults ya estan puestos para una corrida REAL. David solo tiene que hacer Run All.
# (Para un smoke rapido de 2 min, cambia RUN_PROFILE a "debug".)
RUN_PROFILE = "serious"      # "serious" (corrida real, DEFAULT) | "debug" (smoke rapido)

# --- tu arquitectura y el algoritmo ---
ARCH_VARIANT = "positional_v1_1"   # "original_v1_0" | "positional_v1_1" | "mlp_baseline"
ALGORITHM    = "PPO"               # "PPO" | "RecurrentPPO" | "SAC" | "QRDQN" | "A2C" | "DQN"
FRAME_STACK  = 8                   # nº de frames apilados que ve el transformer (8 = una campaña)
FEATURES_DIM = 120

# --- celda de régimen a atacar (rho90_share90 = donde tu DMLPA quedo mas cerca, +0.0034) ---
CELL_ID = "rho90_share90"    # "rho75_share90" | "rho90_share75" | "rho90_share90"

# --- tapes (namespace de DESARROLLO, disjunto de las seeds selladas) ---
TRAIN_TAPE_START = 970_000_000
EVAL_TAPE_START  = 980_000_000
SEED = 42                    # para conclusiones, repite con varias seeds y agrega

# --- presupuestos (serious = 200k, el MISMO presupuesto que uso RecurrentPPO, comparacion justa) ---
TOTAL_TIMESTEPS_DEBUG, EVAL_TAPES_DEBUG     = 5_000,   8
TOTAL_TIMESTEPS_SERIOUS, EVAL_TAPES_SERIOUS = 200_000, 64

# OJO: el codigo del entorno de Program Q vive en esta rama (no en main).
GIT_URL, GIT_BRANCH = "https://github.com/Thom-320/scres-ia.git", "qr1-c1-natural-continuation"

if RUN_PROFILE == "debug":
    TOTAL_TIMESTEPS, EVAL_TAPES = TOTAL_TIMESTEPS_DEBUG, EVAL_TAPES_DEBUG
elif RUN_PROFILE == "serious":
    TOTAL_TIMESTEPS, EVAL_TAPES = TOTAL_TIMESTEPS_SERIOUS, EVAL_TAPES_SERIOUS
else:
    raise ValueError("RUN_PROFILE debe ser 'debug' o 'serious'")

TRAIN_TAPE_END = TRAIN_TAPE_START + max(TOTAL_TIMESTEPS // 8 + 50_000, 50_000)
EVAL_TAPE_END  = EVAL_TAPE_START + EVAL_TAPES - 1
print({"profile": RUN_PROFILE, "arch": ARCH_VARIANT, "algo": ALGORITHM, "cell": CELL_ID,
       "frame_stack": FRAME_STACK, "timesteps": TOTAL_TIMESTEPS, "eval_tapes": EVAL_TAPES})

## 2) Repositorio + dependencias (Colab / Kaggle / local)

In [ ]:

import os, sys, shutil, subprocess, json, math, time
from pathlib import Path
import numpy as np, pandas as pd

IN_COLAB  = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
PKGS = ["simpy>=4.1","numpy>=1.26","gymnasium>=0.29","stable-baselines3>=2.3",
        "sb3-contrib>=2.3","torch>=2.1","einops>=0.7","pandas>=2.2"]

def run(cmd, cwd=None):
    print("$", " ".join(map(str,cmd)))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print((r.stdout or "")[-1500:]); print((r.stderr or "")[-1500:])
    if r.returncode: raise RuntimeError(f"failed: {cmd}")

if IN_KAGGLE:  REPO = Path("/kaggle/working/scresia")
elif IN_COLAB: REPO = Path("/content/scresia")
else:          REPO = Path.cwd()

if IN_COLAB or IN_KAGGLE:
    run([sys.executable,"-m","pip","install","-q",*PKGS])
    shutil.rmtree(REPO, ignore_errors=True)
    run(["git","clone","--depth","1","--branch",GIT_BRANCH,GIT_URL,str(REPO)])
sys.path.insert(0, str(REPO))
print("repo:", REPO, "| exists:", (REPO/"supply_chain").exists())

## 3) El benchmark sellado que hay que superar (constantes)  ·  ⛔ **NO CAMBIAR**

In [ ]:

# ⛔ NO CAMBIAR: resultados SELLADOS que hay que batir.
# Cifras selladas de Submission A (source_of_truth.json, veredicto
# STOP_Q_NO_REPLICATED_LEARNED_ADAPTATION). H_learned = RL - mejor lazo abierto;
# H_neural = RL - mejor clasico. Estas son las cifras a batir.
FROZEN_RECURRENTPPO = {
 "rho75_share90": dict(H_learned=+0.07952, H_learned_lcb=+0.06608, H_neural=-0.00159,
                       H_neural_ci=(-0.00627,+0.00310), favorable_vs_classical_pct=84.8, seeds="10/10"),
 "rho90_share75": dict(H_learned=+0.07255, H_learned_lcb=+0.06233, H_neural=-0.00072,
                       H_neural_ci=(-0.00552,+0.00408), favorable_vs_classical_pct=89.8, seeds="10/10"),
 "rho90_share90": dict(H_learned=+0.11724, H_learned_lcb=+0.10614, H_neural=-0.00041,
                       H_neural_ci=(-0.00268,+0.00186), favorable_vs_classical_pct=95.7, seeds="10/10"),
}
# Tu corrida DMLPA previa (ppo_dmlpa_positional), para referencia.
FROZEN_DMLPA_PREV = {
 "rho75_share90": dict(H_learned=+0.096991, H_neural=-0.006885),
 "rho90_share75": dict(H_learned=+0.051559, H_neural=-0.014526),
 "rho90_share90": dict(H_learned=+0.090935, H_neural=+0.003438),
}
WIN_BAR_H_NEURAL_LCB = 0.01   # SESOI: prima neural real requiere LCB95(H_neural) >= +0.01
WORST_PRODUCT_MARGIN = -0.02  # guardrail de equidad
print("Objetivo en", CELL_ID, "-> superar H_neural de RecurrentPPO =",
      FROZEN_RECURRENTPPO[CELL_ID]["H_neural"], "con la barra LCB95 >= +0.01")

## 4) Entorno Program Q + wrappers  ·  ⛔ **NO CAMBIAR** (es el mismo env que RecurrentPPO)

In [ ]:

# ⛔ NO CAMBIAR: entorno OFICIAL de Program Q (el mismo que enfrento RecurrentPPO)
#    + wrappers (frame-stack para el transformer; Box->argmax para SAC). Cambiarlo
#    rompe la comparacion apples-to-apples.
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

from supply_chain.program_o_ret_env import ProgramORetOnlyEnv, ProgramORetCell
from supply_chain.program_o_full_des_transducer import (
    extract_full_des_skeleton, full_action_calendars, simulate_full_des_frontier)
from supply_chain.program_o_state_rich import StateRichConfiguration, state_rich_calendar

def load_scheduler():
    parent = json.loads((REPO/"contracts/program_o_full_des_hpi_translation_v1.json").read_text())
    return parent["action"]["within_week_schedulers"][parent["action"]["primary_scheduler"]]
SCHEDULER = load_scheduler()

CELL = {c.cell_id: c for c in
        (ProgramORetCell("rho75_share90",0.75,0.90),
         ProgramORetCell("rho90_share75",0.90,0.75),
         ProgramORetCell("rho90_share90",0.90,0.90))}[CELL_ID]

# SAC es continuo; envolvemos Discrete(4) como Box(4) y hacemos argmax.
class BoxToDiscrete(gym.ActionWrapper):
    def __init__(self, env, n=4):
        super().__init__(env)
        self.action_space = spaces.Box(-1.0, 1.0, (n,), np.float32)
    def action(self, a):  return int(np.argmax(np.asarray(a).ravel()))

CONTINUOUS = {"SAC"}   # algoritmos que requieren accion continua

def make_base_env(tape_start, tape_end):
    env = ProgramORetOnlyEnv(scheduler=SCHEDULER, tape_seed_start=tape_start,
                             tape_seed_end=tape_end, cells=[CELL])
    if ALGORITHM in CONTINUOUS:
        env = BoxToDiscrete(env, n=4)
    return Monitor(env)

def make_vec(tape_start, tape_end):
    venv = DummyVecEnv([lambda: make_base_env(tape_start, tape_end)])
    return VecFrameStack(venv, n_stack=FRAME_STACK)   # obs -> FRAME_STACK*21

train_env = make_vec(TRAIN_TAPE_START, TRAIN_TAPE_END)
obs = train_env.reset()
print("obs apilada:", obs.shape, "| accion:", train_env.action_space)
assert obs.shape[1] % FRAME_STACK == 0

## 5) Tu arquitectura  ·  ✅ **EDITA Y REVISA TU ARQUITECTURA AQUÍ**
`DMLPAPositionalV11` es tu transformer con encoding posicional. Cámbiala libremente: es un
`BaseFeaturesExtractor` estándar de SB3. Se inyecta idéntica en TODOS los algoritmos.

In [ ]:

# =========================================================================
# ✅ EDITA / REVISA TU ARQUITECTURA AQUI. Unico requisito para que sea valida en
#    TODOS los algoritmos:  __init__(self, observation_space, factor, features_dim=120)
#    y  forward(x) -> tensor de forma [batch, features_dim].
#    El chequeo de forma al final de la celda lo verifica automaticamente.
# =========================================================================
import torch
from einops import rearrange
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class DMLPAOriginal(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.factor = factor
        self.obs_dimension = observation_space.shape[0] // factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension,100), torch.nn.GELU(),
            torch.nn.Linear(100,features_dim))
        layer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=12, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(layer, num_layers=4)
    def forward(self, x):
        x = rearrange(x, "b (d k) -> b d k", d=self.factor)
        x = self.latent_rw(x); x = self.accumulated(x)
        return x[:, -1, :]

class DMLPAPositionalV11(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.factor = factor
        self.obs_dimension = observation_space.shape[0] // factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension,100), torch.nn.GELU(),
            torch.nn.Linear(100,features_dim))
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        layer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=12, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(layer, num_layers=4)
        self.register_buffer("pe", self._pe(factor, features_dim))
    @staticmethod
    def _pe(seq_len, d):
        pe = torch.zeros(seq_len, d); pos = torch.arange(0,seq_len).unsqueeze(1)
        div = torch.exp(torch.arange(0,d,2)*(-math.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        return pe.unsqueeze(0)
    def forward(self, x):
        x = rearrange(x, "b (d k) -> b d k", d=self.factor)
        x = self.latent_rw(x) + self.pe; x = self.pre_norm(x); x = self.accumulated(x)
        return x[:, -1, :]

ARCHS = {"original_v1_0": DMLPAOriginal, "positional_v1_1": DMLPAPositionalV11, "mlp_baseline": None}
ARCH = ARCHS[ARCH_VARIANT]
# comprobacion de forma
if ARCH is not None:
    _ex = ARCH(train_env.observation_space, factor=FRAME_STACK, features_dim=FEATURES_DIM)
    with torch.no_grad():
        _o = torch.as_tensor(train_env.reset()).float()
        print("arquitectura OK, salida:", _ex(_o).shape)

## 6) Entrenador multi-algoritmo  ·  ✅ hiperparámetros ajustables (mantén presupuesto comparable)
Un solo builder para **PPO, RecurrentPPO, SAC, QR-DQN, A2C, DQN**, todos con TU extractor.
SAC usa el wrapper continuo. RecurrentPPO usa política LSTM (tu transformer va en el extractor).

In [ ]:

from stable_baselines3 import PPO, A2C, DQN, SAC
from sb3_contrib import RecurrentPPO, QRDQN

def policy_kwargs():
    if ARCH is None: return {}
    return dict(features_extractor_class=ARCH,
                features_extractor_kwargs=dict(features_dim=FEATURES_DIM, factor=FRAME_STACK))

def build_model(env):
    pk = policy_kwargs()
    common = dict(policy_kwargs=pk, device="auto", verbose=1, seed=SEED)
    if ALGORITHM == "PPO":
        return PPO("MlpPolicy", env, n_steps=512, batch_size=64, n_epochs=10, gamma=0.99, **common)
    if ALGORITHM == "A2C":
        return A2C("MlpPolicy", env, n_steps=16, gamma=0.99, **common)
    if ALGORITHM == "RecurrentPPO":
        return RecurrentPPO("MlpLstmPolicy", env, n_steps=512, batch_size=64, n_epochs=10,
                            gamma=0.99, **common)
    if ALGORITHM == "DQN":
        return DQN("MlpPolicy", env, learning_starts=1000, buffer_size=50_000,
                   train_freq=8, gamma=0.99, **common)
    if ALGORITHM == "QRDQN":
        return QRDQN("MlpPolicy", env, learning_starts=1000, buffer_size=50_000,
                     train_freq=8, gamma=0.99, **common)
    if ALGORITHM == "SAC":
        # Discrete relajado a Box; SAC es una relajacion (documentada) para explorar.
        return SAC("MlpPolicy", env, learning_starts=1000, buffer_size=50_000,
                   train_freq=1, gamma=0.99, **common)
    raise ValueError(f"algoritmo no soportado: {ALGORITHM}")

model = build_model(train_env)
IS_RECURRENT = ALGORITHM == "RecurrentPPO"
t0 = time.time(); model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("entrenamiento (s):", round(time.time()-t0,1))

## 7) Líneas base (frontera 65 536 + clásico)  ·  ⛔ **NO CAMBIAR** (rivales oficiales)
Para cada tape de evaluación calculamos: (a) el mejor de los **65 536** calendarios de lazo abierto,
y (b) el **clásico belief-MPC**. Así `H_learned` y `H_neural` de tu modelo son justos y self-consistentes.

In [ ]:

# ⛔ NO CAMBIAR: la frontera exacta de 65 536 calendarios y el clasico belief-MPC
#    son los DOS rivales oficiales; se recomputan sobre TUS tapes para una comparacion justa.
_ALL_CALENDARS = full_action_calendars()   # (65536, 8)
print("frontera:", _ALL_CALENDARS.shape[0], "calendarios de lazo abierto")

def tape_baselines(seed):
    sk,_ = extract_full_des_skeleton(seed=int(seed), scheduler=SCHEDULER,
            regime_persistence=CELL.regime_persistence, dominant_share=CELL.dominant_share,
            downstream_freight_physics_mode="fixed_clock_physical_v1")
    # (a) frontera de lazo abierto
    fr = simulate_full_des_frontier(skeleton=sk, scheduler=SCHEDULER, calendars=_ALL_CALENDARS)
    best_open_loop = float(np.max(fr["ret_visible"]))
    # (b) clasico belief-MPC (parametros de creencia congelados 0.75/0.90, como en el paper)
    cal,_ = state_rich_calendar(skeleton=sk.as_dict(), scheduler=SCHEDULER,
            config=StateRichConfiguration("belief_mpc",3),
            regime_persistence=0.75, dominant_share=0.90)
    cm = simulate_full_des_frontier(skeleton=sk, scheduler=SCHEDULER,
            calendars=np.asarray([cal],dtype=np.uint8))
    return dict(seed=int(seed), best_open_loop=best_open_loop,
                classical=float(cm["ret_visible"][0]),
                classical_worst_product=float(cm["worst_product_fill"][0]),
                skeleton=sk)

EVAL_SEEDS = list(range(EVAL_TAPE_START, EVAL_TAPE_END+1))
print("calculando baselines en", len(EVAL_SEEDS), "tapes (frontera 65k c/u)...")
BASE = {}
for i,s in enumerate(EVAL_SEEDS):
    BASE[s] = tape_baselines(s)
    if (i+1) % max(1,len(EVAL_SEEDS)//4)==0: print(f"  {i+1}/{len(EVAL_SEEDS)}")
print("clasico medio:", round(np.mean([BASE[s]['classical'] for s in EVAL_SEEDS]),5),
      "| mejor-lazo-abierto medio:", round(np.mean([BASE[s]['best_open_loop'] for s in EVAL_SEEDS]),5))

## 8) Evalúa y veredicto  ·  ⛔ **NO CAMBIAR la lógica del veredicto/barra**

In [ ]:

# ⛔ NO CAMBIAR: evaluacion determinista + veredicto (barra de victoria).
def run_policy_on_tape(model, seed):
    """Corre el modelo determinista sobre un tape fijo; devuelve metricas terminales.
    Nota: el VecEnv auto-resetea al terminar el episodio, asi que damos un tape de
    reserva (seed..seed+2); las metricas terminales del tape 'seed' quedan en info
    ANTES del auto-reset, que solo avanza al de reserva y se ignora."""
    def _mk():
        e = ProgramORetOnlyEnv(scheduler=SCHEDULER, tape_seed_start=seed, tape_seed_end=seed+2, cells=[CELL])
        return Monitor(BoxToDiscrete(e,4) if ALGORITHM in CONTINUOUS else e)
    venv = VecFrameStack(DummyVecEnv([_mk]), n_stack=FRAME_STACK)
    obs = venv.reset(); state=None; starts=np.ones((1,),bool); info=[{}]
    done=[False]
    while not done[0]:
        if IS_RECURRENT:
            act,state = model.predict(obs, state=state, episode_start=starts, deterministic=True)
        else:
            act,_ = model.predict(obs, deterministic=True)
        obs,rew,done,info = venv.step(act); starts=done
    m = info[0]["metrics"]
    return dict(ret_visible=float(m["ret_visible"]), worst_product_fill=float(m["worst_product_fill"]),
                lost_orders=float(m["lost_orders"]), unresolved_orders=float(m["unresolved_orders"]),
                calendar=info[0]["calendar"])

rows=[]
for s in EVAL_SEEDS:
    r = run_policy_on_tape(model, s); b = BASE[s]
    rows.append(dict(seed=s, model_ret=r["ret_visible"],
        best_open_loop=b["best_open_loop"], classical=b["classical"],
        H_learned=r["ret_visible"]-b["best_open_loop"],
        H_neural =r["ret_visible"]-b["classical"],
        worst_product_delta=r["worst_product_fill"]-b["classical_worst_product"],
        lost=r["lost_orders"], unresolved=r["unresolved_orders"]))
df = pd.DataFrame(rows)

def boot_lcb(x, n=10000, seed=7):
    x=np.asarray(x); rng=np.random.default_rng(seed)
    return float(np.quantile(rng.choice(x,(n,len(x)),replace=True).mean(1),0.025))

H_learned_mean = df.H_learned.mean(); H_neural_mean = df.H_neural.mean()
H_neural_lcb = boot_lcb(df.H_neural.values)
worst_lcb    = boot_lcb(df.worst_product_delta.values)
fav_classical= float((df.H_neural>0).mean()*100)

rp = FROZEN_RECURRENTPPO[CELL_ID]
print("="*72)
print(f"  TU MODELO:  {ALGORITHM} + {ARCH_VARIANT}   |   celda {CELL_ID}   |   {len(df)} tapes")
print("="*72)
print(f"  H_learned (vs 65 536 lazo abierto):  {H_learned_mean:+.5f}   "
      f"[RecurrentPPO: {rp['H_learned']:+.5f}]")
print(f"  H_neural  (vs clasico belief-MPC) :  {H_neural_mean:+.5f}  (LCB95 {H_neural_lcb:+.5f})   "
      f"[RecurrentPPO: {rp['H_neural']:+.5f}]")
print(f"  favorables vs clasico            :  {fav_classical:.1f}%   "
      f"[RecurrentPPO: {rp['favorable_vs_classical_pct']:.1f}%]")
print(f"  worst-product delta (LCB95)      :  {worst_lcb:+.5f}   (barra: >= {WORST_PRODUCT_MARGIN})")
print("-"*72)
beat_openloop   = H_learned_mean > 0
beat_recurrent  = H_neural_mean  > rp["H_neural"]
beat_classical  = H_neural_mean  > 0
pass_win_bar    = (H_neural_lcb >= WIN_BAR_H_NEURAL_LCB) and (worst_lcb >= WORST_PRODUCT_MARGIN)
print(f"  [{'SI' if beat_openloop else 'NO'}] le ganas al lazo abierto (H_learned>0)")
print(f"  [{'SI' if beat_recurrent else 'NO'}] le ganas a RecurrentPPO (H_neural mayor que el suyo)")
print(f"  [{'SI' if beat_classical else 'NO'}] le ganas al clasico (H_neural>0)")
print(f"  [{'SI' if pass_win_bar else 'NO'}] pasas la BARRA de prima neural (LCB95(H_neural)>=+0.01 y equidad OK)")
print("="*72)
if pass_win_bar:
    print("  *** PRIMA NEURAL REAL — esto seria el primer resultado que le gana limpio al MPC. Sellar y confirmar. ***")
elif beat_classical:
    print("  Le ganas al clasico en media, pero aun no con LCB95>=+0.01. Sube seeds/tapes y re-mide.")
else:
    print("  Todavia no superas al clasico. Igual que RecurrentPPO y tu DMLPA previo: sin prima neural aun.")
display(df.describe()[["model_ret","H_learned","H_neural","worst_product_delta"]])

## 9) Guardar resultados

In [ ]:

out = Path("scresia_david_outputs"); out.mkdir(exist_ok=True)
tag = f"{ALGORITHM.lower()}_{ARCH_VARIANT}_{CELL_ID}_{RUN_PROFILE}"
df.to_csv(out/f"eval_{tag}.csv", index=False)
summary = dict(algorithm=ALGORITHM, arch=ARCH_VARIANT, cell=CELL_ID, profile=RUN_PROFILE,
    n_tapes=len(df), H_learned_mean=H_learned_mean, H_neural_mean=H_neural_mean,
    H_neural_lcb95=H_neural_lcb, worst_product_delta_lcb95=worst_lcb,
    favorable_vs_classical_pct=fav_classical,
    beat_open_loop=bool(beat_openloop), beat_recurrentppo=bool(beat_recurrent),
    beat_classical=bool(beat_classical), pass_win_bar=bool(pass_win_bar),
    recurrentppo_target=FROZEN_RECURRENTPPO[CELL_ID])
(out/f"summary_{tag}.json").write_text(json.dumps(summary, indent=2))
try: model.save(out/f"model_{tag}")
except Exception as e: print("save modelo:", e)
print("guardado en", out.resolve(), "\n", json.dumps({k:summary[k] for k in
      ("H_neural_mean","H_neural_lcb95","beat_recurrentppo","pass_win_bar")}, indent=2))

---
### Notas honestas para David
- **Barra correcta:** la victoria no es `H_neural>0` por ruido; es **LCB95(H_neural) ≥ +0.01** con
  guardrails. Es exactamente lo que RecurrentPPO **no** logró (empató en ~0). Si tu arquitectura lo
  logra de forma estable, es publicable.
- **SAC** aquí es una **relajación**: la acción discreta (4 opciones) se expone como `Box(4)` y se toma
  `argmax`. Es legítimo para explorar, pero para un claim final conviene un off-policy discreto (QR-DQN)
  o dejar SAC como ablación.
- **`serious`:** sube `RUN_PROFILE="serious"`, corre **varias seeds** (cambia `SEED`) y agrega, porque
  una sola corrida no distingue señal de ruido. RecurrentPPO usó 10 seeds × 256 tapes.
- **Comparación justa:** este notebook recomputa la frontera y el clásico sobre TUS tapes, así que tu
  `H_neural` es directamente comparable al de RecurrentPPO (ambos = learner − clásico).
- Un claim SELLADO requeriría el protocolo prospectivo (seeds selladas, contrato congelado). Este
  notebook es para **explorar y descubrir** si tu arquitectura tiene la señal; si la tiene, lo
  formalizamos.
